# DGM-HMamba — GPU experiments

Runs the real-data experiment (Indian Pines → Pavia University) with a chosen configuration on a Colab GPU and prints the same summary as the CPU experiments: 5-way 1-/5-shot OA and AA on the meta-testing pool, the full 9-class map, validation accuracy (a pool held out from the meta-training pool) and training accuracy.

*Runtime → Change runtime type → T4 GPU*, edit `ARGS` below if needed, then *Runtime → Run all*. Code branch: `exp/gpu`.

In [ ]:
import os, subprocess, json

BRANCH = "exp/gpu"
ARGS = ("--val-fraction 0.1 --val-episodes 40 --val-every 250 --select-on-val "
        "--schedule cosine --warmup-frac 0.05 --stage1-iters 1000 --stage2-lr 2e-3 "
        "--train-way 9 --train-query 10 --augment --stage2-steps 3000 "
        "--patch-size 9 --widths 64,64,64 --expand 2 --ema 0.99 --no-diffusion "
        "--grad-checkpoint --map-repeats 20 --seed 42")

def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {cmd}")

if not os.path.isdir("DGM-HMamba"):
    run(f"git clone -q -b {BRANCH} https://github.com/zarrinmahdavipour/DGM-HMamba.git")
os.chdir("DGM-HMamba")
run(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}")
run("pip -q install -r requirements.txt")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
run("python scripts/prepare_data.py --data-root data --download indian_pines pavia_u")
run(f"python scripts/demo_real.py --data-root data --output-dir runs/gpu {ARGS}")

In [ ]:
s = json.load(open("runs/gpu/demo_summary.json")); h = s["history"]
for k, m in s["episodic_metrics"].items():
    print(f"RESULT {k}: OA {m['oa_mean']*100:.2f} +/- {m['oa_std']*100:.2f}  AA {m['aa']*100:.2f}  kappa {m['kappa']:.4f}")
mm = s["map_metrics"]
print(f"RESULT map ({mm['repeats']} draws): OA {mm['oa_mean']*100:.2f} +/- {mm['oa_sd']*100:.2f}  AA {mm['aa_mean']*100:.2f} +/- {mm['aa_sd']*100:.2f}")
r1 = lambda xs: [None if x is None else round(x * 100, 1) for x in xs]
print("RESULT map recall   :", r1(mm["recall_mean"]))
print("RESULT map precision:", r1(mm["pooled_precision"]))
print("RESULT map F1       :", r1(mm["pooled_f1"]))
print("RESULT validation", {k: round(v*100, 2) for k, v in h["validation"].items()})
print("RESULT train acc (last 10%):", round(h["train_acc_last10pct"]*100, 2), "| selected episode:", h["selected_episode"],
      "| skipped:", h.get("skipped_updates"))
print("RESULT split", s["split_info"])
print("RESULT params", s["trainable_parameters"], "| timings", {k: round(v, 2) for k, v in s["timings_s"].items()})
from IPython.display import Image, display
display(Image("runs/gpu/demo_maps.png")); display(Image("runs/gpu/demo_curves.png"))